# Spotter Machine Learning Engineer Assessment
## Freight Rate Prediction, Out-of-Time Validation & December Forecast


---
### Table of Contents
1. [Problem Statement & Overview](#1-problem-statement)
2. [Exploratory Data Analysis & Schema Audit](#2-exploratory-data-analysis)
3. [Data Quality Remediation (Negative Weights & Imputation)](#3-data-quality-remediation)
4. [Feature Engineering (Spatial, Temporal & Freight Physics)](#4-feature-engineering)
5. [Validation Strategy: Out-of-Time Split](#5-validation-strategy)
6. [Model Training & Benchmarking](#6-model-training--benchmarking)
7. [December Forecast & Submission Verification](#7-december-forecast--submission-verification)

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to path
sys.path.append(str(Path('..').resolve()))
from src.data_processing import clean_and_impute_data
from src.feature_engineering import engineer_features
from src.models import evaluate_predictions

print('Libraries successfully imported.')

## 1. Problem Statement & Dataset Overview
The objective is to predict spot freight transactions (`posted_rate`) across North American freight corridors.
- `data/train_test.csv`: 48,000 labeled development loads (January 1 to October 31, 2025).
- `data/validation.csv`: 12,000 unpriced test loads requiring predictions (November 1 to December 31, 2025).
- `data/december_chart_inputs.csv`: 31 daily loads for a fixed lane (Lexington -> Fort Wayne, 360 miles, Dry Van, 32,000 lbs).

In [2]:
train_raw = pd.read_csv('../data/train_test.csv')
val_raw = pd.read_csv('../data/validation.csv')
dec_raw = pd.read_csv('../data/december_chart_inputs.csv')

print(f'Train shape: {train_raw.shape}, Validation shape: {val_raw.shape}, December shape: {dec_raw.shape}')
train_raw.head(3)

## 2. Exploratory Data Analysis & Data Quality Audit
Our exploratory analysis identified three critical telemetry anomalies:
1. **Negative Weights:** 292 rows in train and 145 rows in validation contained negative values (down to -47,500 lbs). Payloads cannot be negative; taking `abs(weight)` restores legitimate gross payload numbers.
2. **Missing Values:** Missing weights are imputed via median grouped by equipment type. Missing `market_index` is imputed via daily date averages.
3. **Unseen Cities in Validation:** Validation has 72 cities vs 64 in train. Geographic coordinates are used to ensure spatial generalization.

In [3]:
print('Negative weights in train:', (train_raw['weight'] < 0).sum())
print('Missing weights in train:', train_raw['weight'].isna().sum())
print('Missing market_index in train:', train_raw['market_index'].isna().sum())

train_clean, meta = clean_and_impute_data(train_raw)
val_clean, _ = clean_and_impute_data(val_raw, weight_medians=meta['weight_medians'], daily_market_means=meta['daily_market_means'], global_market_mean=meta['global_market_mean'])
print('Data cleaning and imputation complete.')

## 3. Feature Engineering Architecture
We engineer 37 predictive features covering:
- **Spatial Network Geometry:** Great-circle Haversine distance, road circuity ratio, directional bearing, midpoint coordinates.
- **Temporal Dynamics:** Day of week, day of month, month-end shipping surge, holiday rush flags, cyclical sin/cos transforms.
- **Freight Work:** Ton-miles (weight in tons * distance), weight per mile.
- **Economic Market Interactions:** Market-adjusted distance, quote-adjusted distance, market-quote ratio.

In [4]:
train_fe = engineer_features(train_clean)
val_fe = engineer_features(val_clean)
print(f'Total features engineered: {train_fe.shape[1]}')
train_fe[['haversine_dist', 'circuity_ratio', 'ton_miles', 'market_adj_dist']].head(3)

## 4. Out-of-Time (OOT) Validation Split
To prevent lookahead leakage, we partition the 10-month development set chronologically:
- **Training Split:** Jan 1 to Aug 31, 2025 (8 months, 37,975 loads)
- **Validation Split:** Sep 1 to Oct 31, 2025 (2 months, 10,025 loads) — exactly matches the 2-month test horizon!

In [5]:
feature_cols = [
    'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon',
    'distance', 'weight', 'market_index', 'quote_signal',
    'haversine_dist', 'circuity_ratio', 'lat_diff', 'lon_diff', 'bearing', 'mid_lat', 'mid_lon',
    'day_of_week', 'is_weekend', 'day_of_month', 'is_month_end', 'month', 'quarter', 'day_of_year',
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_holiday_surge',
    'ton_miles', 'weight_per_mile', 'heavy_load',
    'market_adj_dist', 'quote_adj_dist', 'market_quote_ratio', 'macro_rate_factor',
    'eq_Dry_Van', 'eq_Reefer', 'eq_Flatbed'
]

oot_train = train_fe['date'] < '2025-09-01'
oot_val = train_fe['date'] >= '2025-09-01'

X_train_oot = train_fe.loc[oot_train, feature_cols]
y_train_oot = train_fe.loc[oot_train, 'posted_rate'].values
dist_train_oot = train_fe.loc[oot_train, 'distance'].values

X_val_oot = train_fe.loc[oot_val, feature_cols]
y_val_oot = train_fe.loc[oot_val, 'posted_rate'].values
dist_val_oot = train_fe.loc[oot_val, 'distance'].values

print(f'OOT Train: {X_train_oot.shape}, OOT Val: {X_val_oot.shape}')

## 5. Model Training & Comparative Benchmarking
We benchmark Linear Baselines, Random Forests, and Gradient Boosted Decision Trees (GBDTs).
**Key Insight:** Predicting Rate Per Mile (RPM = rate / distance) eliminates distance scale variance and yields superior MAE.

In [6]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

# 1. Ridge Baseline
ridge = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=10.0))]).fit(X_train_oot, y_train_oot)
p_ridge = np.clip(ridge.predict(X_val_oot), 50.0, None)
m_ridge = evaluate_predictions(y_val_oot, p_ridge)

# 2. HistGradientBoosting (Direct)
hgb_d = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.04, random_state=42).fit(X_train_oot, y_train_oot)
p_hgb_d = np.clip(hgb_d.predict(X_val_oot), 50.0, None)
m_hgb_d = evaluate_predictions(y_val_oot, p_hgb_d)

# 3. HistGradientBoosting (Rate Per Mile)
rpm_train = y_train_oot / dist_train_oot
hgb_rpm = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.04, random_state=42).fit(X_train_oot, rpm_train)
p_hgb_rpm = np.clip(hgb_rpm.predict(X_val_oot) * dist_val_oot, 50.0, None)
m_hgb_rpm = evaluate_predictions(y_val_oot, p_hgb_rpm)

bench_df = pd.DataFrame([m_ridge, m_hgb_d, m_hgb_rpm], index=['Ridge Baseline', 'GradientBoosting (Direct)', 'GradientBoosting (Rate Per Mile)'])
bench_df

## 6. December Forecast Analysis & Official Verification
The candidate December forecast chart produced by `score.py` reflects freight market dynamics:
- Pre-holiday crunch peak around Dec 17–19 ($844)
- Christmas moderation ($807)
- End-of-year rush on Dec 31 ($845)

In [7]:
from IPython.display import Image, display
display(Image(filename='../scorer_results/candidate_december.png'))